# 11 · Control-tower AI/BI dashboard

Builds and deploys the **Supply Chain Control Tower** Lakeview dashboard: cross-twin KPI counters,
per-domain bar charts, and detail tables over the twins. **Every dataset query is tested first.**

Source of truth for the dashboard — re-run to (re)create + publish.

## Config

In [ ]:
dbutils.widgets.text("target_catalog", "mickmickstableawscoff_catalog", "Target catalog")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")
dbutils.widgets.text("warehouse_id", "", "SQL warehouse id (blank = first running)")
dbutils.widgets.text("parent_path", "", "Workspace folder (blank = home/agentic-supply-chain)")
from databricks.sdk import WorkspaceClient
_w = WorkspaceClient()
C = f"{dbutils.widgets.get('target_catalog').strip()}.{dbutils.widgets.get('monitoring_schema').strip()}"
WID = dbutils.widgets.get('warehouse_id').strip()
if not WID:
    _whs = list(_w.warehouses.list())
    WID = next((x.id for x in _whs if str(x.state).endswith('RUNNING')), _whs[0].id)
PARENT = dbutils.widgets.get('parent_path').strip() or f"/Workspace/Users/{_w.current_user.me().user_name}/agentic-supply-chain"
print("datasets over:", C, "| warehouse:", WID, "| folder:", PARENT)

## Datasets (tested before deploy)

In [ ]:
DATASETS = {
  "overview": ("Cross-twin KPIs",
    f"SELECT (SELECT COUNT(*) FROM {C}.inventory_twin_current WHERE is_stockout OR is_low_cover) AS inv_at_risk, "
    f"(SELECT COUNT(*) FROM {C}.procurement_twin_current WHERE supplier_risk_flag) AS suppliers_at_risk, "
    f"(SELECT COUNT(*) FROM {C}.production_twin_current WHERE is_bottleneck) AS bottlenecks, "
    f"(SELECT COUNT(*) FROM {C}.distribution_twin_current WHERE fulfillment_risk_flag) AS fulfillment_at_risk, "
    f"(SELECT SUM(backlog_qty) FROM {C}.demand_twin_current) AS total_backlog, "
    f"(SELECT AVG(otif_rate) FROM {C}.procurement_twin_current) AS avg_supplier_otif, "
    f"(SELECT AVG(fill_rate) FROM {C}.distribution_twin_current) AS avg_fill_rate, "
    f"(SELECT AVG(forecast_wmape) FROM {C}.demand_twin_current) AS avg_wmape"),
  "demand_by_group": ("Demand by material group",
    f"SELECT material_group, SUM(backlog_qty) AS backlog, SUM(open_demand_qty) AS open_demand, "
    f"SUM(horizon_total_forecast_qty) AS horizon_forecast FROM {C}.demand_twin_current GROUP BY material_group ORDER BY backlog DESC"),
  "inventory_by_group": ("Inventory by material group",
    f"SELECT material_group, SUM(on_hand_qty) AS on_hand, AVG(days_of_cover_demand) AS avg_days_cover, "
    f"SUM(CASE WHEN is_stockout OR is_low_cover THEN 1 ELSE 0 END) AS at_risk_positions FROM {C}.inventory_twin_current GROUP BY material_group ORDER BY on_hand DESC"),
  "procurement_supplier": ("Suppliers",
    f"SELECT supplier_name, AVG(otif_rate) AS otif, AVG(avg_lead_time_days) AS lead_days, SUM(open_po_qty) AS open_po "
    f"FROM {C}.procurement_twin_current GROUP BY supplier_name ORDER BY open_po DESC LIMIT 12"),
  "production_material": ("Production by material",
    f"SELECT material_text, AVG(yield_pct) AS yield_pct, AVG(scrap_pct) AS scrap_pct, SUM(open_production_qty) AS open_prod "
    f"FROM {C}.production_twin_current GROUP BY material_text ORDER BY open_prod DESC LIMIT 12"),
  "distribution_channel": ("Distribution by channel",
    f"SELECT channel, AVG(fill_rate) AS fill_rate, AVG(otif_rate) AS otif, AVG(avg_delay_days) AS delay_days, SUM(delivered_qty) AS delivered "
    f"FROM {C}.distribution_twin_current GROUP BY channel ORDER BY delivered DESC"),
  "demand_backlog_top": ("Top backlog series",
    f"SELECT material_text, plant, channel, open_demand_qty, backlog_qty, forecast_wmape "
    f"FROM {C}.demand_twin_current WHERE backlog_qty > 0 ORDER BY backlog_qty DESC LIMIT 20"),
}
for name, (disp, sql) in DATASETS.items():
    n = spark.sql(sql).count()
    print(f"{name}: {n} rows OK")

## Build dashboard JSON

In [ ]:
import json
def dset(name):
    disp, sql = DATASETS[name]; return {"name": name, "displayName": disp, "queryLines": [sql]}
def text(name, mds, x, y, w, h):
    return {"widget": {"name": name, "multilineTextboxSpec": {"lines": [mds]}}, "position": {"x": x, "y": y, "width": w, "height": h}}
def counter(name, dataset, field, title, x, y, w=3, h=3):
    return {"widget": {"name": name, "queries": [{"name": "main_query", "query": {"datasetName": dataset,
        "fields": [{"name": field, "expression": f"`{field}`"}], "disaggregated": True}}],
        "spec": {"version": 2, "widgetType": "counter", "encodings": {"value": {"fieldName": field, "displayName": title}},
        "frame": {"title": title, "showTitle": True}}}, "position": {"x": x, "y": y, "width": w, "height": h}}
def bar(name, dataset, xf, yf, xlab, ylab, title, x, y, w=6, h=5):
    return {"widget": {"name": name, "queries": [{"name": "main_query", "query": {"datasetName": dataset,
        "fields": [{"name": xf, "expression": f"`{xf}`"}, {"name": yf, "expression": f"`{yf}`"}], "disaggregated": True}}],
        "spec": {"version": 3, "widgetType": "bar", "encodings": {
            "x": {"fieldName": xf, "scale": {"type": "categorical"}, "displayName": xlab},
            "y": {"fieldName": yf, "scale": {"type": "quantitative"}, "displayName": ylab}},
        "frame": {"title": title, "showTitle": True}}}, "position": {"x": x, "y": y, "width": w, "height": h}}
def table(name, dataset, cols, title, x, y, w=12, h=6):
    return {"widget": {"name": name, "queries": [{"name": "main_query", "query": {"datasetName": dataset,
        "fields": [{"name": c0, "expression": f"`{c0}`"} for c0, _ in cols], "disaggregated": True}}],
        "spec": {"version": 2, "widgetType": "table", "encodings": {"columns": [{"fieldName": c0, "displayName": c1} for c0, c1 in cols]},
        "frame": {"title": title, "showTitle": True}}}, "position": {"x": x, "y": y, "width": w, "height": h}}

layout = [
  text("title", "## Supply Chain Control Tower", 0, 0, 12, 1),
  text("subtitle", f"Cross-twin KPIs and risk from `{C}`.", 0, 1, 12, 1),
  counter("kpi-inv-risk", "overview", "inv_at_risk", "Inventory at risk", 0, 2),
  counter("kpi-sup-risk", "overview", "suppliers_at_risk", "Suppliers at risk", 3, 2),
  counter("kpi-bottleneck", "overview", "bottlenecks", "Bottlenecks", 6, 2),
  counter("kpi-fulfil-risk", "overview", "fulfillment_at_risk", "Fulfillment at risk", 9, 2),
  counter("kpi-backlog", "overview", "total_backlog", "Total backlog (qty)", 0, 5),
  counter("kpi-otif", "overview", "avg_supplier_otif", "Avg supplier OTIF", 3, 5),
  counter("kpi-fill", "overview", "avg_fill_rate", "Avg fill rate", 6, 5),
  counter("kpi-wmape", "overview", "avg_wmape", "Avg forecast WMAPE", 9, 5),
  text("h-di", "### Demand & inventory", 0, 8, 12, 1),
  bar("c-demand", "demand_by_group", "material_group", "backlog", "Material group", "Backlog qty", "Demand backlog by material group", 0, 9),
  bar("c-inv", "inventory_by_group", "material_group", "avg_days_cover", "Material group", "Avg days of cover", "Inventory days of cover by group", 6, 9),
  text("h-ppd", "### Procurement, production & distribution", 0, 14, 12, 1),
  bar("c-proc", "procurement_supplier", "supplier_name", "otif", "Supplier", "Avg OTIF", "Supplier OTIF (top by open PO)", 0, 15),
  bar("c-prod", "production_material", "material_text", "yield_pct", "Material", "Avg yield %", "Production yield by material", 6, 15),
  bar("c-dist", "distribution_channel", "channel", "fill_rate", "Channel", "Avg fill rate", "Distribution fill rate by channel", 0, 20),
  table("t-proc", "procurement_supplier", [("supplier_name","Supplier"),("otif","OTIF"),("lead_days","Lead days"),("open_po","Open PO qty")], "Supplier detail", 6, 20, 6, 5),
  text("h-backlog", "### Demand backlog - top series", 0, 25, 12, 1),
  table("t-backlog", "demand_backlog_top", [("material_text","Material"),("plant","Plant"),("channel","Channel"),("open_demand_qty","Open demand"),("backlog_qty","Backlog"),("forecast_wmape","WMAPE")], "Top backlog series", 0, 26, 12, 6),
]
dashboard = {"datasets": [dset(n) for n in DATASETS], "pages": [{"name": "control_tower",
    "displayName": "Control Tower", "pageType": "PAGE_TYPE_CANVAS", "layoutVersion": "GRID_V1", "layout": layout}]}
print("widgets:", len(layout))

## Deploy + publish
Creates the dashboard, then publishes it. Prints the URL.

In [ ]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
body = {"display_name": "Supply Chain Control Tower", "parent_path": PARENT,
        "warehouse_id": WID, "serialized_dashboard": json.dumps(dashboard)}
resp = w.api_client.do("POST", "/api/2.0/lakeview/dashboards", body=body)
did = resp["dashboard_id"]
w.api_client.do("POST", f"/api/2.0/lakeview/dashboards/{did}/published", body={"warehouse_id": WID})
print("dashboard_id:", did)
print("URL:", f"{w.config.host}/dashboardsv3/{did}/published")